This notebook is a study guide for Lectures 15 - 27 for Midterm 2.
It is for self-study and is separate from the class review in L28.

Midterm 2 is a printed exam with no computer access.
You may bring one page (one side) of handwritten notes.
You may need to complete a short line of printed code.

The examples and practice problems here will help you be familiar with the code that is printed on the exam.
You need to be able to read these lines and say what they will do, without access to the Python environment.

You do not need to memorize code syntax except for a few basics (printing, indexing, that kind of thing).

# Lectures index

## Lectures 15-18: loops, conditions, and functions

[Lecture 15](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture15.ipynb):
* `for` loops and `range`
* `if`, `elif`, and `else`

[Lecture 16](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture16.ipynb):
* `and`, `or`, and `not`
* `while`, `break`, and `continue`

[Lecture 17](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture17.ipynb):
* defining a function with `def`
* `print` versus `return`

[Lecture 18](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture18.ipynb):
* returning more than one value
* keyword arguments and default values
* local names and docstrings

## Lectures 19-22: curve fitting, objects, and minimize

[Lecture 19](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture19.ipynb):
* a model function that `curve_fit` calls
* `popt` and the starting guess `p0`

[Lecture 20](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture20.ipynb):
* objects, attributes, and methods
* `type`, `dir`, and the `linregress` result

[Lecture 21](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture21.ipynb):
* objective functions and `x0`
* `res.x`, `res.fun`, and `res.success`

[Lecture 22](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture22.ipynb):
* least squares as an objective
* comparing `minimize` with `curve_fit`

## Lectures 23-27: scoring fits, outliers, and two inputs

[Lecture 23](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture23.ipynb):
* the `rmse(y, y_fit)` score
* comparing fitted forms

[Lecture 24](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture24.ipynb):
* the influence of one reading on least squares
* mean absolute error as an objective

[Lecture 25](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture25.ipynb):
* color as a third variable with `c=` and `colorbar`
* contour maps with `tricontourf`

[Lecture 26](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture26.ipynb):
* fitting a plane with `minimize`
* reading the fitted coefficients

[Lecture 27](https://colab.research.google.com/github/wfreinhart/matse219/blob/main/notebooks/Lecture27.ipynb):
* selecting rows with masks
* RMSE against a mean-only guess
* residuals and predictions

# Solved examples

These are intended as mini tutorials for individual concepts from the class.
Each one is fully worked: read the code and say what it prints.
They use one shared dataset throughout (the concrete mixes from L27).

Each row is one mix.
The ingredient columns are in kg/m$^3$.
`age` is in days and `strength` is in MPa.

## Setup

In [ ]:
#@title Load the concrete data and give the columns short names (click ▶ to run) { display-mode: "form" }
import numpy as np
import pandas as pd
data = pd.read_csv('https://raw.githubusercontent.com/wfreinhart/matse219/main/datasets/concrete.csv')
data.columns = ['cement', 'slag', 'fly_ash', 'water', 'superplasticizer',
                'coarse_agg', 'fine_agg', 'age', 'strength']

### Loading modules

Make sure you know how and when to import a module.
It only needs to be done once in a notebook.
One strategy is to do it near the beginning.

> On the exam we will just assume these are all imported!

In [ ]:
import numpy as np                          # numpy: array math and statistics functions
import pandas as pd                         # pandas: tables (DataFrames)
from matplotlib import pyplot as plt        # the plotting submodule
from scipy import stats, optimize           # the module with curve_fit and minimize

### Selecting the mixes

Slag and fly ash replace part of the cement.
We keep the plain-cement mixes, the ones with none of either.
`&` keeps the rows where both masks are `True`.

In [ ]:
is_plain = (data['slag'] == 0) & (data['fly_ash'] == 0)    # True where both are zero
is_28 = data['age'] == 28                                  # True for the 28-day tests

plain = data.loc[is_plain]                  # plain-cement mixes of every age
mixes = data.loc[is_plain & is_28]          # plain-cement mixes tested at 28 days
print(len(plain), len(mixes))               # rows in each selection

## Lectures 15-18: loops, conditions, and functions

### Functions with defaults and return values

A function stores a recipe under a name.
A `return` value becomes the value of the call.
A default in the `def` line makes an input optional.

The binder is all the cementing material.

In [ ]:
def mix_ratios(cement, water, slag=0, fly_ash=0):
    """Water/binder ratio and total binder in kg/m3."""
    binder = cement + slag + fly_ash   # all the cementing material
    ratio = water / binder             # water per unit of binder
    return ratio, binder               # two values come back, in this order

Call it with the defaults. Then call it with a keyword argument:

In [ ]:
ratio, binder = mix_ratios(350, 180)   # slag and fly_ash use their defaults
print(ratio, binder)                   # the two returned values

ratio, binder = mix_ratios(350, 180, fly_ash=50)   # a keyword argument names the input
print(ratio, binder)                               # binder now includes the fly ash

Parameters without defaults come first.

In [ ]:
def ratio_only(cement, water, limit=0.5):
    ratio = water / cement                       # water per unit of cement
    return ratio, ratio <= limit                 # the ratio and whether it is within the limit

Call it with the default limit. Then name the input to change it:

In [ ]:
ratio, within = ratio_only(350, 180)             # limit uses its default of 0.5
print(ratio, within)

ratio, within = ratio_only(350, 180, limit=0.4)  # name the input to change the default
print(ratio, within)

### `print` versus `return`

`print` shows a value on the screen.
`return` gives the value to the caller.
A function that returns its value can be used in later math.

In [ ]:
def get_ratio(cement, water):
    return water / cement            # hand the value back to the caller

ratio_now = get_ratio(350, 180)      # the call is now a number
print(2 * ratio_now)                 # doubling works because the value came back

### Loops with `if`, `elif`, `else`, `break`, and `continue`

A loop over positions reads parallel lists together.
`.tolist()` turns a column into a list.

In [ ]:
strength_list = mixes['strength'].tolist()   # MPa, one entry per mix
cement_list = mixes['cement'].tolist()       # kg/m3
water_list = mixes['water'].tolist()         # kg/m3
print(len(strength_list))                    # number of mixes

Label the first six mixes by strength:

In [ ]:
for i in range(6):                           # the first six mixes
    if strength_list[i] >= 50:
        label = 'high'                       # only the first True branch runs
    elif strength_list[i] >= 35:             # checked only when the line above was False
        label = 'medium'
    else:
        label = 'low'                        # runs when no condition was True
    print(i, label)

A counter starts at zero before the loop and changes inside it.

In [ ]:
count_high = 0                                 # start the count at zero
for strength_i in strength_list:
    if strength_i >= 50:
        count_high = count_high + 1            # inside the if, inside the for
print(count_high)                              # after the loop, so it prints once

`break` ends the loop.

In [ ]:
for i in range(len(water_list)):
    ratio, binder = mix_ratios(cement_list[i], water_list[i])   # the function inside the loop
    if ratio > 0.5:
        print(i, ratio)                        # the first mix above 0.5
        break                                  # nothing after this mix is checked
print('stopped at position', i)

`continue` skips to the next pass.

In [ ]:
strongest = 0
for i in range(len(strength_list)):
    if water_list[i] > 200:
        continue                               # skip this mix and go to the next one
    if strength_list[i] > strongest:
        strongest = strength_list[i]           # the largest strength among mixes with water <= 200
print(strongest)

## Lectures 19-22: curve fitting, objects, and minimize

### Objects: attributes and methods

An attribute is a stored value and takes no parentheses.
A method is a function attached to the object and takes parentheses.

In [ ]:
cement = mixes['cement'].to_numpy()          # kg/m3, as a NumPy array
water = mixes['water'].to_numpy()            # kg/m3
strength = mixes['strength'].to_numpy()      # MPa

Look at the attributes and methods of the array:

In [ ]:
print(type(strength))                        # the kind of object
print(strength.shape)                        # an attribute: no parentheses
print(strength.mean())                       # a method, so it takes parentheses
print(strength.mean)                         # no parentheses gives the method itself, not a number
print('shape' in dir(strength))              # dir lists the names attached to the object

`stats.linregress` returns one object with named attributes.

In [ ]:
fit = stats.linregress(cement, strength)     # strength as a straight line in cement
print(type(fit))                             # the kind of object
print(fit.slope)                             # MPa per kg/m3 of cement
print(fit.intercept)                         # the line's value at zero cement
print(fit.rvalue)                            # an attribute, so no parentheses

### Fitting a model with `curve_fit`

Strength grows with age and levels off.
A model function takes `x` first and the parameters after it.
`curve_fit(model, x, y, p0=...)` returns `popt`.
`popt` holds the best-fit values in the order of the `def` line.
`p0` gives one starting value per parameter, in the same order.

In [ ]:
age_all = plain['age'].to_numpy()            # days
strength_all = plain['strength'].to_numpy()  # MPa

def age_model(age, s_max, rate):
    """Strength in MPa from s_max * (1 - exp(-rate * age)); age is in days."""
    return s_max * (1 - np.exp(-rate * age))   # levels off at s_max for long ages

Fit the model to the plain-cement mixes:

In [ ]:
popt, pcov = optimize.curve_fit(age_model, age_all, strength_all, p0=[40, 0.1])
s_max, rate = popt                           # unpacked in the order of the def line
print(popt)                                  # s_max in MPa, rate in 1/day

Predict the strength at four ages:

In [ ]:
for age in [3, 7, 28, 90]:                   # a loop over four ages
    print(age, age_model(age, *popt))        # * unpacks popt into the two parameters

Plot the data and the fitted curve:

In [ ]:
age_line = np.linspace(1, 365, 200)          # 200 evenly spaced ages
strength_line = age_model(age_line, *popt)   # the fitted curve at those ages

fig, ax = plt.subplots()
ax.plot(age_all, strength_all, '.', label='data')       # one marker per mix
ax.plot(age_line, strength_line, '-', label='fit')      # draw the fitted curve
ax.set_xlabel('age (days)')
ax.set_ylabel('strength (MPa)')
ax.legend()
plt.show()

### Fitting with `minimize`

The objective takes the parameters and returns one number: the sum of squared residuals.
`minimize(objective, x0)` calls it with trial values.
`x0` and `res.x` follow the unpack order inside the objective.

In [ ]:
def sse_age(params):
    s_max, rate = params                                         # one value per parameter
    residuals = strength_all - age_model(age_all, s_max, rate)   # data minus model
    return np.sum(residuals**2)                                  # one number comes back

print(sse_age([40, 0.1]))                    # the objective at one trial point
print(sse_age([20, 0.5]))                    # the objective at another trial point

Minimize it and compare with `curve_fit`:

In [ ]:
res_age = optimize.minimize(sse_age, [40, 0.1])
print(res_age.x)                             # the best parameters, as an array
print(res_age.fun)                           # the objective at that point
print(res_age.success)                       # solver status
print(popt)                                  # curve_fit minimizes the same sum

## Lectures 23-27: scoring fits, outliers, and two inputs

### RMSE and residuals

RMSE is the typical miss in the units of the data.
A model that always guesses the mean sets the baseline.

In [ ]:
def rmse(y, y_fit):
    """Root-mean-square error, in the units of y."""
    residuals = y - y_fit                      # measured minus predicted
    return np.sqrt(np.mean(residuals**2))

Score the fit and the mean-only guess:

In [ ]:
strength_fit = age_model(age_all, *res_age.x)    # fitted strength for every mix
print(rmse(strength_all, strength_fit))          # the model's typical miss in MPa
print(rmse(strength_all, np.mean(strength_all))) # the mean-only guess for every mix

Plot the residuals against the fitted strength:

In [ ]:
residuals = strength_all - strength_fit          # measured minus fitted

fig, ax = plt.subplots()
ax.scatter(strength_fit, residuals, color='k')   # one marker per mix
ax.axhline(0, color='k')                         # zero residual
ax.set_xlabel('fitted strength (MPa)')
ax.set_ylabel('residual (MPa)')
plt.show()

### Outliers and the MAE objective

One bad reading can pull a least-squares fit.
The mean absolute error (MAE) grows linearly with the size of a residual.
The squared error grows quadratically.
We plant a typo in a copy of the 28-day strengths.

In [ ]:
fit_clean = stats.linregress(cement, strength)   # least squares on the recorded values

bad_index = np.argmax(strength)                  # the position of the largest strength
strength_typo = strength.copy()                  # a copy, so strength stays unchanged
strength_typo[bad_index] = 10 * strength_typo[bad_index]   # planted typo: one extra zero
print(strength[bad_index], strength_typo[bad_index])       # recorded value and typo value

Fit the typo values with least squares:

In [ ]:
fit_typo = stats.linregress(cement, strength_typo)   # least squares on the typo values
print(fit_clean.slope, fit_typo.slope)               # the typo moves the slope

Define the MAE objective for a line:

In [ ]:
def mae_line(params):
    m, b = params                                          # slope and intercept
    residuals = strength_typo - (m * cement + b)           # strength_typo comes from the notebook
    return np.mean(np.abs(residuals))                      # absolute value first, then the mean

Minimize it:

In [ ]:
res_mae = optimize.minimize(mae_line, [0.1, 10])
m_mae, b_mae = res_mae.x
print(m_mae, b_mae)                                        # minimum-MAE slope and intercept
print(res_mae.fun, res_mae.success)                        # the MAE and the solver status

`success` is False here because the solver stopped with a precision-loss message.
The line it returns is still close to the minimum-MAE line.

### Color and contour plots

The mixes with slag and no fly ash form a new table.
`c=` gives `ax.scatter` one number per marker.
`cmap` turns the numbers into colors.
`fig.colorbar` needs the object that `ax.scatter` returns.
`ax.tricontourf(x, y, z)` fills bands of equal `z` and interpolates between the points.

In [ ]:
is_blend = (data['slag'] > 0) & (data['fly_ash'] == 0)        # slag but no fly ash
blend = data.loc[is_blend & is_28]                            # 28-day slag mixes
slag = blend['slag'].to_numpy()                               # kg/m3
water_blend = blend['water'].to_numpy()                       # kg/m3
strength_blend = blend['strength'].to_numpy()                 # MPa
print(len(blend))                                             # mixes in the table

Color each mix by its strength:

In [ ]:
fig, ax = plt.subplots()
im = ax.scatter(slag, water_blend, c=strength_blend, cmap='viridis')   # one marker per mix
cb = fig.colorbar(im, ax=ax)                                           # the colorbar reads the colors from im
cb.set_label('28-day strength (MPa)')                                  # names the colorbar
ax.set_xlabel('slag (kg/m3)')
ax.set_ylabel('water (kg/m3)')
plt.show()

Fill bands of equal strength:

In [ ]:
fig, ax = plt.subplots()
cs = ax.tricontourf(slag, water_blend, strength_blend, cmap='viridis')   # filled bands
ax.scatter(slag, water_blend, color='white', edgecolor='k')              # the measured mixes on top
cb = fig.colorbar(cs, ax=ax)                                             # the colorbar takes the contour object
cb.set_label('28-day strength (MPa)')
ax.set_xlabel('slag (kg/m3)')
ax.set_ylabel('water (kg/m3)')
plt.show()

### Fitting a plane with `minimize`

A plane in two inputs has three parameters:

$$\text{strength} = a + b\,(\text{slag}) + c\,(\text{water})$$

The model takes the inputs first and the parameters after them.
`x0` has one start value per parameter.
Each coefficient is the change in fitted strength in MPa for one more kg/m$^3$ of that ingredient.

In [ ]:
def plane(slag, water, a, b, c):
    """Strength in MPa from slag and water, both in kg/m3."""
    return a + b * slag + c * water

def mse_plane(params):
    a, b, c = params                                                     # one value per parameter
    residuals = strength_blend - plane(slag, water_blend, a, b, c)       # data minus plane
    return np.mean(residuals**2)

Minimize from zeros and read the coefficients:

In [ ]:
res_plane = optimize.minimize(mse_plane, [0, 0, 0])
a_fit, b_fit, c_fit = res_plane.x                        # unpacked in the order of the unpack line
print(res_plane.success)
print(res_plane.x)                                       # a, b, c
print(b_fit, c_fit)                                      # MPa per kg/m3 of slag, then of water

Score the plane against the mean-only guess.

In [ ]:
strength_plane = plane(slag, water_blend, *res_plane.x)   # fitted strength for every mix
print(rmse(strength_blend, strength_plane))               # the plane's typical miss
print(rmse(strength_blend, np.mean(strength_blend)))      # the mean-only guess

# Practice problems

These are designed to focus on individual concepts from lecture and reinforce your fundamental knowledge.
They use the same `data`, `plain`, and `mixes` tables as the solved examples.
`np.log(age)` and `.copy()` are supplied where a problem needs them.

## Lectures 15-18: loops, conditions, and functions

### Functions

(a) Write a function `water_cement_ratio(water, cement)` that returns the water divided by the cement.
Call it for 180 kg/m$^3$ of water and 350 kg/m$^3$ of cement.

(b) Write a function `strength_in_psi(strength_mpa, factor=145.04)` that returns the strength multiplied by `factor`.
Call it once with the default and once with `factor=145`.

### Loops and conditions

(a) Build a list of the `strength` values of `mixes`.
Count how many are above 40 MPa, using a counter.

(b) For each mix, print `'low'` below 30 MPa.
Print `'medium'` from 30 up to 50 MPa.
Print `'high'` otherwise.

(c) Print the position of the first mix whose strength is below 40 MPa.
End the loop there.

(d) Find the largest `strength` among the mixes with `water` at or below 180 kg/m$^3$, using `continue` to skip the others.

## Lectures 19-22: curve fitting, objects, and minimize

### Objects

(a) Print the `shape` and the `type` of the `strength` array from `mixes`.

(b) Print the mean of the array two ways: with a method and with a NumPy function.

(c) Fit a line to strength against `water` with `stats.linregress`.
Print the slope, the intercept, and `rvalue`.

### `curve_fit`

(a) Write a model function `log_model(age, a, b)` that returns `a * np.log(age) + b`.

(b) Fit it to `plain['age']` and `plain['strength']` with `curve_fit` and print `popt`.

(c) Print the fitted strength at 7, 28, and 90 days using `*popt`.

(d) Plot the data and the fitted curve on the same figure.
Add axis labels and a legend.

(e) Plot the residuals of the fit against `age`.
Read off the age range where the fitted curve sits furthest from the data.

### `minimize`

(a) Write an objective `mse_log(params)` that returns the mean of the squared residuals of `log_model` for the plain-cement mixes of every age.

(b) Call `optimize.minimize` with a starting guess of `[5, 10]`.
Print `res.x`, `res.fun`, and `res.success`.

(c) Print `res.x` next to the `popt` from `curve_fit`.

(d) Write an objective for one variable `x` that returns `(x - 3)**2`.
Minimize it from `x0=10` and print `res.x[0]`.

## Lectures 23-27: scoring fits, outliers, and two inputs

### RMSE and residuals

(a) Compute the RMSE of `log_model` with the fitted parameters.
Compute the RMSE of the mean-only guess.

(b) Print the largest absolute residual of `log_model` and the position where it occurs.

### MAE objective

(a) Write an objective `mae_water(params)` that returns the mean absolute residual of the line `m * water + b` for `strength` in `mixes`.

(b) Score two candidate lines with it: `m=-0.5, b=140` and `m=-0.64, b=155`.
Print both values.

### Fitting a plane

(a) Write a function `plane_fine(cement, fine_agg, a, b, c)` that returns `a + b * cement + c * fine_agg`.

(b) Write an objective for it using the `mixes` columns `cement`, `fine_agg`, and `strength`.

(c) Fit it with `minimize` and an `x0` of three entries.
Print the three fitted values.

(d) Use the fitted `b` to calculate the change in fitted strength when cement rises by 50 kg/m$^3$ with fine aggregate fixed.
Check it by calling `plane_fine` at two points.

(e) Compute the RMSE of the plane and of the mean-only guess.
Print the fraction by which the plane lowers the RMSE.

(f) Predict the strength at the mean of both inputs.
Mark that mix on a filled contour plot of the fitted plane.